# Taller Interactivo: Operación Registraduría (Introducción a MapReduce)

**Contexto:** Es el dia de las elecciones presidenciales en Colombia. Los candidatos principales son: **Cepeda, Paloma, Fajardo y Abelardo**. 

A nivel nacional, hay mas de 100,000 mesas de votacion (nuestros Ayudantes de Cocina). Si intentamos enviar cada papelito (voto) en camioneta hasta la Registraduria Central (Memoria RAM) en Bogota, el sistema colapsara y los resultados tardaran semanas.

Vamos a implementar el verdadero paradigma del **Big Data: MapReduce** usando la libreria profesional `mrjob`.

---
## 🧠 La Teoria: ¿Que es exactamente MapReduce?

Olvidate de la programacion clasica (`for i in datos:`). En un cluster de 1,000 computadores, el codigo funciona en dos fases magicas:

1. **Fase MAP (El Jurado de Mesa):** Un computador lee un pedacito de los datos (un voto). Extrae lo que importa y **emite** una pareja de datos llamada `(Clave, Valor)`. Ej: `('Fajardo', 1)`.
   * **Fase Oculta (Shuffle & Sort):** El sistema viaja por la red y junta magicamente todos los Valores que comparten la misma Clave: `('Fajardo', [1, 1, 1, 1...])`.
2. **Fase REDUCE (El Escrutador Central):** Otro computador recibe esa lista agrupada por candidato y aplica una operacion estadistica (suma, maximo, etc.).

Para simular que programamos muchos computadores a la vez en este Notebook, usaremos el comando magico `%%writefile nombre.py`. Esto guarda el codigo en un archivo real, y luego usaremos la consola Linux de Codespaces `uv run nombre.py datos.txt` para ejecutarlo.

In [1]:
# Verificamos que 'uv' instalo mrjob correctamente en nuestro entorno de GitHub Codespaces
!uv pip list | grep mrjob
print('Si ves mrjob en la lista, el entorno esta listo.')

Using Python 3.12.1 environment at: /workspaces/ClaseBigData/.venv
mrjob                   0.7.4
Si ves mrjob en la lista, el entorno esta listo.


### 🏭 Fabrica de Votos Sinteticos (Ejecuta esta celda)
Vamos a generar un archivo simulado con cientos de miles de votos para nuestros ejercicios.

In [4]:
import random

candidatos = ['Cepeda', 'Paloma', 'Fajardo', 'Abelardo', 'MickeyMouse', '']

# 1. Generamos votos individuales sueltos (Urna Nacional)
with open('votos_sueltos.txt', 'w') as f:
    for _ in range(50000):
        f.write(random.choice(candidatos) + '\n')

# 2. Generamos reportes de actas de mesa (Candidato, Departamento, Votos en esa mesa)
with open('votos_actas.txt', 'w') as f:
    for _ in range(10000):
        cand = random.choice(candidatos[:4]) # Solo candidatos validos aqui
        dept = random.choice(['Antioquia', 'Cundinamarca', 'Valle', 'Atlantico', 'Santander'])
        votos_mesa = random.randint(10, 500)
        f.write(f'{cand},{dept},{votos_mesa}\n')

print('Archivos creados: votos_sueltos.txt y votos_actas.txt')

Archivos creados: votos_sueltos.txt y votos_actas.txt


---
## 🟢 EJERCICIO 1 (10 Puntos): Tu Primer Escrutinio Distribuido
Queremos contar cuantos votos tiene cada candidato en `votos_sueltos.txt`. 
- **Map:** Por cada linea (voto) que leas, debes gritar (yield) el nombre del candidato y un 1.
- **Reduce:** Recibiras el nombre del candidato y una lista/iterador de muchisimos unos. Debes sumarlos.

In [2]:
%%writefile ejercicio1.py
from mrjob.job import MRJob

class EscrutinioNacional(MRJob):

    def mapper(self, _, linea):
        voto = linea.strip()
        # INICIA TU CODIGO AQUI
        # Usa la palabra reservada 'yield' para emitir la pareja (Clave, Valor)
        # La Clave es la variable 'voto', el Valor es el numero 1
        yield voto, 1
        # TERMINA TU CODIGO AQUI

    def reducer(self, candidato, conteos):
        # INICIA TU CODIGO AQUI
        # 'conteos' es un generador con todos los 1s juntos: [1, 1, 1...]
        # Calcula la suma y emite la Clave (candidato) y el Valor final (la suma)
        total = sum(conteos) # Pista: usa sum()
        yield candidato, total
        # TERMINA TU CODIGO AQUI

if __name__ == '__main__':
    EscrutinioNacional.run()

Writing ejercicio1.py


In [7]:
!uv pip install setuptools

Using Python 3.12.1 environment at: /workspaces/ClaseBigData/.venv
Resolved 1 package in 92ms                                           
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/982.64 KiB          
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/982.64 KiB        
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/982.64 KiB        
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/982.64 KiB        
⠙ Preparing packages... (0/1)------------------- 63.52 KiB/982.64 KiB        
⠙ Preparing packages... (0/1)------------------- 79.52 KiB/982.64 KiB        
⠙ Preparing packages... (0/1)------------------- 95.52 KiB/982.64 KiB        
⠙ Preparing packages... (0/1)------------------- 111.52 KiB/982.64 KiB       
⠙ Preparing packages... (0/1)------------------- 127.52 KiB/982.64 KiB       
⠙ Preparing packages... (0/1)------------------- 143.52 KiB/982.64 KiB       
⠙ 

In [8]:
!uv run ejercicio1.py votos_sueltos.txt
# Si el codigo de arriba es correcto, veras los totales de cada candidato en la consola.

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/ejercicio1.codespace.20260316.224403.208162
Running step 1 of 1...
job output is in /tmp/ejercicio1.codespace.20260316.224403.208162/output
Streaming final output from /tmp/ejercicio1.codespace.20260316.224403.208162/output...
""	8400
"Abelardo"	8370
"Cepeda"	8392
"Fajardo"	8224
"MickeyMouse"	8434
"Paloma"	8180
Removing temp directory /tmp/ejercicio1.codespace.20260316.224403.208162...


---
## 🟡 EJERCICIO 2 (10 Puntos): MapReduce con Filtro de Origen
En los datos de arriba viste que "MickeyMouse" y votos en blanco ('') sacaron miles de votos.
**Concepto Clave de Big Data:** NUNCA envies datos basura por la red hacia el Reducer. ¡Consume el ancho de banda del cluster! El filtrado (limpieza) debe hacerse directo en el **Mapper**.

In [9]:
%%writefile ejercicio2.py
from mrjob.job import MRJob

class EscrutinioLimpio(MRJob):

    def mapper(self, _, linea):
        voto = linea.strip()
        candidatos_oficiales = ['Cepeda', 'Paloma', 'Fajardo', 'Abelardo']
        
        # INICIA TU CODIGO AQUI
        # Pon un 'if' para asegurar que el 'voto' esta dentro de 'candidatos_oficiales'
        # Solo si se cumple, haz el yield.
        if voto in candidatos_oficiales:
            yield voto, 1
        pass
        # TERMINA TU CODIGO AQUI

    def reducer(self, candidato, conteos):
        yield candidato, sum(conteos)

if __name__ == '__main__':
    EscrutinioLimpio.run()

Writing ejercicio2.py


In [10]:
!uv run ejercicio2.py votos_sueltos.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/ejercicio2.codespace.20260316.224700.492371
Running step 1 of 1...
job output is in /tmp/ejercicio2.codespace.20260316.224700.492371/output
Streaming final output from /tmp/ejercicio2.codespace.20260316.224700.492371/output...
"Abelardo"	8370
"Cepeda"	8392
"Fajardo"	8224
"Paloma"	8180
Removing temp directory /tmp/ejercicio2.codespace.20260316.224700.492371...


---
## 🟠 EJERCICIO 3 (10 Puntos): El "Combiner" (Optimizacion extrema)
Piensa en esto: Si una mesa tiene 5,000 votos por Paloma, el Mapper envia 5,000 mensajes independientes de `('Paloma', 1)` por internet. El cable de fibra optica del pais va a explotar.

**Solucion: El Combiner.** Es un "Mini-Reducer" que hace una pre-suma local en el computador del Jurado de Mesa, ANTES de enviar los datos por internet. Envia `('Paloma', 5000)`.

In [11]:
%%writefile ejercicio3.py
from mrjob.job import MRJob

class EscrutinioEficiente(MRJob):

    def mapper(self, _, linea):
        voto = linea.strip()
        if voto in ['Cepeda', 'Paloma', 'Fajardo', 'Abelardo']:
            yield voto, 1

    def combiner(self, candidato, conteos_locales):
        # INICIA TU CODIGO AQUI
        # La logica del Combiner para sumas algebraicas... ¡Es idéntica a la del reducer!
        # Suma los conteos locales y envialos (haz yield)
        yield candidato, sum(conteos_locales)
        # TERMINA TU CODIGO AQUI

    def reducer(self, candidato, conteos_totales):
        yield candidato, sum(conteos_totales)

if __name__ == '__main__':
    EscrutinioEficiente.run()

Writing ejercicio3.py


In [12]:
!uv run ejercicio3.py votos_sueltos.txt
# El resultado es identico matematicamente, pero a nivel de hardware local en el Cluster, acabas de ahorrar un 99% de ancho de banda en la red.

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/ejercicio3.codespace.20260316.225123.957752
Running step 1 of 1...
job output is in /tmp/ejercicio3.codespace.20260316.225123.957752/output
Streaming final output from /tmp/ejercicio3.codespace.20260316.225123.957752/output...
"Abelardo"	8370
"Cepeda"	8392
"Fajardo"	8224
"Paloma"	8180
Removing temp directory /tmp/ejercicio3.codespace.20260316.225123.957752...


---
## 🔴 EJERCICIO 4 (10 Puntos): Mas alla de la Suma (Maximo Distribuido)
Dejamos los votos sueltos. Ahora usamos las ACTAS por mesa (`votos_actas.txt`). 
El formato de cada linea de texto es `Candidato,Departamento,Votos_Numero`.
**Reto:** No queremos sumar. Queremos saber ¿cual fue el Record Maximo (pico mas alto de votos) que saco cada candidato en una sola acta/mesa?

In [13]:
%%writefile ejercicio4.py
from mrjob.job import MRJob

class MaximoPorCandidato(MRJob):

    def mapper(self, _, linea):
        partes = linea.strip().split(',')
        if len(partes) == 3:
            candidato = partes[0]
            votos = int(partes[2])  # Cuidado: convertir texto a entero
            
            # INICIA TU CODIGO AQUI
            # Emite el candidato y los votos de esta acta especifica
            yield candidato, votos
            # TERMINA TU CODIGO AQUI

    def reducer(self, candidato, votos_de_todas_las_mesas):
        # INICIA TU CODIGO AQUI
        # Usa la funcion estadistica max() para iterar y sacar el maximo
        record = max(votos_de_todas_las_mesas)
        yield candidato, record
        # TERMINA TU CODIGO AQUI

if __name__ == '__main__':
    MaximoPorCandidato.run()

Writing ejercicio4.py


In [14]:
!uv run ejercicio4.py votos_actas.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/ejercicio4.codespace.20260316.225411.046918
Running step 1 of 1...
job output is in /tmp/ejercicio4.codespace.20260316.225411.046918/output
Streaming final output from /tmp/ejercicio4.codespace.20260316.225411.046918/output...
"Abelardo"	500
"Cepeda"	500
"Fajardo"	500
"Paloma"	500
Removing temp directory /tmp/ejercicio4.codespace.20260316.225411.046918...


---
## 🟣 EJERCICIO 5 (10 Puntos): Resolviendo la Falacia del Promedio
En la clase anterior demostramos que NO puedes calcular promedios promediando resultados previos.
Debes pedirle a los ayudantes (Mappers) que te den los *Ingredientes Intermedios*. 

**Reto:** Calcular el Promedio Historico de Votos por Acta para cada candidato.

In [1]:
%%writefile ejercicio5.py
from mrjob.job import MRJob

class PromedioActas(MRJob):

    def mapper(self, _, linea):
        partes = linea.strip().split(',')
        if len(partes) == 3:
            candidato = partes[0]
            votos = int(partes[2])
            
            # INICIA TU CODIGO AQUI
            # Para reconstruir el promedio, el Mapper debe entregar DOS valores como tupla: (la suma, el conteo)
            # Ej: Emite los votos leídos, y el numero 1 para ir contando de acta en acta.
            yield candidato, (votos, 1)
            # TERMINA TU CODIGO AQUI

    def reducer(self, candidato, valores_tuplas):
        suma_total = 0
        cantidad_mesas = 0
        
        # INICIA TU CODIGO AQUI
        # Desempacamos la tupla (votos, 1) que llego del Mapper
        for votos, un_acta in valores_tuplas:
            suma_total += votos  # Suma los votos
            cantidad_mesas += un_acta # Suma las actas (unos)
        
        promedio = suma_total / cantidad_mesas
        yield candidato, round(promedio, 2)
        # TERMINA TU CODIGO AQUI

if __name__ == '__main__':
    PromedioActas.run()

Writing ejercicio5.py


In [2]:
!python ejercicio5.py votos_actas.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/ejercicio5.codespace.20260321.145058.278827
Running step 1 of 1...
job output is in /tmp/ejercicio5.codespace.20260321.145058.278827/output
Streaming final output from /tmp/ejercicio5.codespace.20260321.145058.278827/output...
"Paloma"	253.41
"Cepeda"	256.68
"Fajardo"	253.76
"Abelardo"	253.35
Removing temp directory /tmp/ejercicio5.codespace.20260321.145058.278827...


---
## 🌟 BONO FINAL PARA ESTADÍSTICOS (10 Puntos): Varianza Distribuida

La logica nos dice que la formula de Varianza Poblacional $Var = \frac{\sum (X - \mu)^2}{N}$ necesita el Promedio Global ($\mu$) primero. ¡Pero en Big Data no puedes pasar los datos por la red dos veces!

El truco es la manipulacion algebraica (Esperanza de X al cuadrado):
$Var = \frac{\sum X^2}{N} - (\frac{\sum X}{N})^2$

¡Solo necesitamos pasar TRES valores en la tupla: **Conteo, X y X-cuadrado**!

In [5]:
%%writefile bono_varianza.py
from mrjob.job import MRJob

class VarianzaActas(MRJob):

    def mapper(self, _, linea):
        partes = linea.strip().split(',')
        if len(partes) == 3:
            cand = partes[0]
            x = float(partes[2])
            
            # INICIA TU CODIGO AQUI
            # Emite una tupla con 3 valores: (El numero 1, el valor original 'x', y el valor al cuadrado)
            yield cand, (1, x, x**2)
            # TERMINA TU CODIGO AQUI

    def reducer(self, candidato, tuplas):
        n = 0
        suma_x = 0
        suma_x2 = 0
        
        for conteo, x, x2 in tuplas:
            n += conteo
            suma_x += x
            suma_x2 += x2
            
        # INICIA TU CODIGO AQUI
        if n > 0:
            promedio = suma_x / n
            # Implementa la formula matematica descrita en el markdown usando las variables ya acumuladas arriba
            varianza = (suma_x / n) - (promedio ** 2)
            yield candidato, round(varianza, 2)
        # TERMINA TU CODIGO AQUI

if __name__ == '__main__':
    VarianzaActas.run()

Overwriting bono_varianza.py


In [6]:
!python bono_varianza.py votos_actas.txt
print('\nSi llegaste hasta aqui y los calculos de varianza son razonables... ¡Felicidades! Tienes cerebro de Data Engineer Creador de Modelos.')

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/bono_varianza.codespace.20260321.145958.559680
Running step 1 of 1...
job output is in /tmp/bono_varianza.codespace.20260321.145958.559680/output
Streaming final output from /tmp/bono_varianza.codespace.20260321.145958.559680/output...
"Paloma"	-63964.79
"Cepeda"	-65625.44
"Fajardo"	-64138.88
"Abelardo"	-63935.03
Removing temp directory /tmp/bono_varianza.codespace.20260321.145958.559680...

Si llegaste hasta aqui y los calculos de varianza son razonables... ¡Felicidades! Tienes cerebro de Data Engineer Creador de Modelos.
